# Custom De-Identification Pipeline

| Field | Behavior |
|---|---|
| **PATIENT name** | Replaced with Patient ID from lookup map |
| **DOCTOR name** | Kept as-is (not de-identified) |
| **DATE** (service dates etc.) | Reduced to Month + Year only (e.g. `April 2026`) |
| **DOB** (date of birth) | Converted to Age (e.g. `53 years old`) |
| **ZIP / Postcode** | First 3 chars kept, rest replaced with X (e.g. `944XX`) |

## 1. Setup

In [ ]:
import json, os

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@11"

with open("spark_jsl.json") as f:
    license_keys = json.load(f)

locals().update(license_keys)
os.environ.update(license_keys)
print("License keys loaded. JSL_VERSION:", license_keys.get("JSL_VERSION"))

## 2. Imports & Spark

In [ ]:
import sparknlp
import sparknlp_jsl

from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.pretrained import PretrainedPipeline

import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.ml import Pipeline, PipelineModel

import pandas as pd

spark = sparknlp_jsl.start(license_keys["SECRET"])
print("Spark NLP Version      :", sparknlp.version())
print("Spark NLP JSL Version  :", sparknlp_jsl.version())

## 3. Configuration
### Patient ID Lookup
Add patient name → patient ID mappings here (or load from your DB).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PATIENT ID LOOKUP  — populate from your database
# ─────────────────────────────────────────────────────────────────────────────
PATIENT_ID_MAP = {
    "Daniel Foster" : "PT-10042",
    "Laura Foster"  : "PT-10043",
    "John Smith"    : "PT-20017",
    "Noah Lee"      : "PT-30085",
    # Add more:  "Full Patient Name": "PT-XXXXX"
}

def get_patient_id(name: str) -> str:
    """Return patient ID from map, or a hash-based placeholder if unknown."""
    return PATIENT_ID_MAP.get(name, f"PT-{abs(hash(name)) % 90000 + 10000}")

print("Patient map loaded:", PATIENT_ID_MAP)

## 4. Helper Transformation Functions

In [ ]:
import re
from datetime import datetime, date as date_type

# All date formats we try to parse
_DATE_FORMATS = [
    "%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d",
    "%m-%d-%Y", "%d-%m-%Y",
    "%d.%m.%Y", "%m.%d.%Y",
    "%B %d, %Y", "%b %d, %Y",
    "%d %B %Y", "%d %b %Y",
    "%Y/%m/%d",
]

def _parse_date(s: str):
    for fmt in _DATE_FORMATS:
        try:
            return datetime.strptime(s.strip(), fmt)
        except ValueError:
            pass
    return None

# ── DATE → Month Year ──────────────────────────────────────────────────────
def date_to_month_year(date_str: str) -> str:
    """04/21/2026  →  April 2026"""
    dt = _parse_date(date_str)
    if dt:
        return dt.strftime("%B %Y")
    # Fallback: extract bare year if nothing else parses
    m = re.search(r'\b(19|20)\d{2}\b', date_str)
    return m.group(0) if m else "[DATE]"

# ── DOB → Age ──────────────────────────────────────────────────────────────
def dob_to_age(dob_str: str) -> str:
    """04/11/1972  →  53 years old"""
    dt = _parse_date(dob_str)
    if dt:
        today = date_type.today()
        age = today.year - dt.year - ((today.month, today.day) < (dt.month, dt.day))
        if 0 <= age <= 120:
            return f"{age} years old"
    return "[DOB]"

# ── ZIP → Partial mask ─────────────────────────────────────────────────────
def partial_zip(zip_str: str) -> str:
    """94404 → 944XX  |  M13 9PL → M13XXXX"""
    z = zip_str.strip()
    keep = 3
    if len(z) > keep:
        return z[:keep] + "X" * (len(z) - keep)
    return z

# Quick sanity check
print(date_to_month_year("04/21/2026"))  # April 2026
print(dob_to_age("04/11/1972"))           # 53 years old
print(partial_zip("94404"))               # 944XX
print(partial_zip("M13 9PL"))             # M13XXXX

## 5. Build NER Pipeline Stages

In [ ]:
# ── Base stages ────────────────────────────────────────────────────────────
documentAssembler = (
    DocumentAssembler()
    .setInputCol("text")
    .setOutputCol("document")
)

splitter = (
    InternalDocumentSplitter()
    .setInputCols("document")
    .setOutputCol("sentence")
    .setSplitMode("recursive")
    .setSplitPatterns([r"\s+|(?<=\G.{512})"])
    .setPatternsAreRegex(True)
    .setChunkSize(512)
    .setChunkOverlap(50)
    .setEnableSentenceIncrement(True)
)

tokenizer = (
    Tokenizer()
    .setInputCols("sentence")
    .setOutputCol("token")
)

tokenizer_doc = (
    Tokenizer()
    .setInputCols("document")
    .setOutputCol("token_doc")
)

In [ ]:
# ── ZeroShot NER (detects PATIENT, DOCTOR, DATE, DATE_OF_BIRTH, ZIP, etc.) ─
# NOTE: DATE_OF_BIRTH is mapped to DOB (not DATE) in the merge step below
labels = [
    "DOCTOR", "PATIENT", "DATE_OF_BIRTH", "DATE",
    "CITY", "STREET", "STATE", "COUNTRY",
    "PHONE", "EMAIL", "ZIP", "USERNAME",
    "ID", "BIOID", "ORGANIZATION", "MEDICAL_RECORD_NUMBER", "SSN", "AGE",
]

zero_shot_ner = (
    PretrainedZeroShotNERChunker
    .pretrained("zeroshot_ner_deid_subentity_docwise_medium", "en", "clinical/models")
    .setInputCols("sentence")
    .setOutputCol("ner_zero_shot")
    .setPredictionThreshold(0.7)
    .setLabels(labels)
    .setBatchSize(8)
)

In [ ]:
# ── Rule-based annotators ──────────────────────────────────────────────────
zip_parser = (
    ContextualParserModel
    .pretrained("zip_parser", "en", "clinical/models")
    .setInputCols(["document", "token_doc"])
    .setOutputCol("zip_chunks")
)

dob_parser = (
    ContextualParserModel
    .pretrained("date_of_birth_parser", "en", "clinical/models")
    .setInputCols(["document", "token_doc"])
    .setOutputCol("dob_chunks")
)

email_matcher = (
    RegexMatcherInternalModel
    .pretrained("email_matcher", "en", "clinical/models")
    .setInputCols(["document"])
    .setOutputCol("email_chunks")
)

country_matcher = (
    TextMatcherInternalModel
    .pretrained("country_matcher", "en", "clinical/models")
    .setInputCols(["document", "token_doc"])
    .setOutputCol("country_chunks")
    .setMergeOverlapping(True)
)

In [ ]:
# ── Merge NER chunks ────────────────────────────────────────────────────────
#   DATE_OF_BIRTH → DOB  (so age calculation applies, not month-year)
#   MEDICAL_RECORD_NUMBER → MEDICALRECORD
chunk_merge_ner = (
    ChunkMergeModel()
    .setInputCols("ner_zero_shot")
    .setOutputCol("ner_merged")
    .setMergeOverlapping(True)
    .setSelectionStrategy("DiverseLonger")
    .setResetSentenceIndices(True)
    .setReplaceDict({
        "DATE_OF_BIRTH"        : "DOB",          # → age
        "MEDICAL_RECORD_NUMBER": "MEDICALRECORD",
    })
)

# Merge rule-based annotators (dob_parser already outputs entity=DOB)
chunk_merge_rules = (
    ChunkMergeModel()
    .setInputCols("zip_chunks", "email_chunks", "dob_chunks", "country_chunks")
    .setOutputCol("rules_merged")
    .setMergeOverlapping(True)
    .setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"])
    .setSelectionStrategy("Sequential")
)

# Final merge: NER wins over rule-based on overlap
chunk_merge_final = (
    ChunkMergeModel()
    .setInputCols("ner_merged", "rules_merged")
    .setOutputCol("ner_chunk")
    .setMergeOverlapping(True)
    .setResetSentenceIndices(True)
    .setOrderingFeatures(["ChunkBegin"])
    .setSelectionStrategy("Sequential")
)

## 6. Custom De-ID Transformation (Spark UDF)
Replaces the standard `LightDeIdentification` with logic tailored to each entity type.

In [ ]:
@F.udf(StringType())
def custom_deid_udf(text, begins, ends, results, metadata_list):
    """
    Apply per-entity custom transformations:
      PATIENT   → Patient ID from lookup
      DOCTOR    → kept as-is
      DATE      → Month Year  (e.g. April 2026)
      DOB       → Age         (e.g. 53 years old)
      ZIP       → Partial     (e.g. 944XX)
      others    → [ENTITY_LABEL]
    """
    if not text or not results:
        return text

    # Build list of (begin, end, entity, chunk_text)
    chunks = []
    for i, chunk_text in enumerate(results):
        b = begins[i] if begins else 0
        e = ends[i]   if ends   else 0
        meta = metadata_list[i] if metadata_list else {}
        entity = meta.get("entity", "") if meta else ""
        chunks.append((b, e, entity, chunk_text))

    # Process from end → start so char positions stay valid
    chunks.sort(key=lambda x: x[0], reverse=True)

    result = text
    for begin, end, entity, chunk_text in chunks:

        if entity == "DOCTOR":
            continue                                         # ← keep as-is

        elif entity == "PATIENT":
            replacement = get_patient_id(chunk_text)        # ← Patient ID

        elif entity == "DATE":
            replacement = date_to_month_year(chunk_text)    # ← Month Year

        elif entity == "DOB":
            replacement = dob_to_age(chunk_text)            # ← Age

        elif entity in ("ZIP", "ZIPCODE"):
            replacement = partial_zip(chunk_text)           # ← 944XX

        else:
            replacement = f"[{entity}]"                     # ← generic mask

        result = result[:begin] + replacement + result[end + 1:]

    return result

print("UDF registered.")

## 7. Assemble & Fit Pipeline

In [ ]:
pipeline = Pipeline(stages=[
    # ── Base ────────────────────────────────────────────────────────
    documentAssembler,
    splitter,
    tokenizer,
    tokenizer_doc,
    # ── NER ─────────────────────────────────────────────────────────
    zero_shot_ner,
    chunk_merge_ner,
    # ── Rule-based ──────────────────────────────────────────────────
    zip_parser,
    dob_parser,
    email_matcher,
    country_matcher,
    chunk_merge_rules,
    # ── Final merge ─────────────────────────────────────────────────
    chunk_merge_final,
])

empty_df = spark.createDataFrame([[""]], ["text"])
pipeline_model = pipeline.fit(empty_df)
print("Pipeline fitted.")

## 8. Test

In [ ]:
sample = """
Patient Mr Daniel Foster (NHS No: 882 441 9930) was brought to St. Mary's Hospital
after collapsing outside his home at 27 Oakfield Road, Manchester, Greater Manchester, M13 9PL.
His wife, Mrs Laura Foster, contacted emergency services using her mobile (07785 441229)
and later emailed from laura.foster@familymail.co.uk.
DOB: 04/11/1972.
On arrival, assessment by Dr Sarah Milton confirmed transient hypotension.
A review of records (IDNUM: MF-22017) showed previous presyncope.
His employer, Greenfield Analytics Ltd, based at 12 Riverbank Street, Leeds, LS8 4HE.
Contact: Mr Oliver Kent, reachable on 0161 882 3300.
Username: dfoster72. Service date: 03/15/2024.
"""

test_df = spark.createDataFrame([[sample]], ["text"])
ner_output = pipeline_model.transform(test_df)

In [ ]:
# Inspect detected entities before applying custom de-ID
ner_output.select(
    F.explode(
        F.arrays_zip(
            ner_output.ner_chunk.result,
            ner_output.ner_chunk.metadata,
        )
    ).alias("cols")
).select(
    F.expr("cols['0']").alias("chunk"),
    F.expr("cols['1']['entity']").alias("entity"),
).show(50, truncate=False)

In [ ]:
# Apply custom de-identification
result_df = ner_output.withColumn(
    "custom_deid_text",
    custom_deid_udf(
        F.col("text"),
        F.col("ner_chunk.begin"),
        F.col("ner_chunk.end"),
        F.col("ner_chunk.result"),
        F.col("ner_chunk.metadata"),
    )
)

pd.set_option("display.max_colwidth", None)
result_df.select("text", "custom_deid_text").toPandas()

## 9. Save & Reload Pipeline

In [ ]:
SAVE_PATH = "content/models/custom_deid_pipeline_v2"
pipeline_model.write().overwrite().save(SAVE_PATH)
print(f"Pipeline saved to: {SAVE_PATH}")

In [ ]:
# To reload later (models already cached, fast load):
# loaded_model = PipelineModel.load(SAVE_PATH)
# result = loaded_model.transform(test_df)
print("To reload: PipelineModel.load(SAVE_PATH)")

## Notes

**Adding more patients to the lookup:**
```python
PATIENT_ID_MAP["Jane Doe"] = "PT-99001"
```

**Loading from a database instead:**
```python
import psycopg2  # or your DB driver
conn = psycopg2.connect(...)
rows = conn.execute("SELECT full_name, patient_id FROM patients").fetchall()
PATIENT_ID_MAP = dict(rows)
```

**Entity list produced by this pipeline:**

| Entity | Treatment |
|--------|----------|
| PATIENT | → Patient ID |
| DOCTOR | → kept as-is |
| DATE | → Month Year |
| DOB | → Age |
| ZIP | → 944XX |
| CITY, STREET, STATE, COUNTRY | → `[CITY]` etc. |
| PHONE, EMAIL | → `[PHONE]`, `[EMAIL]` |
| SSN, ID, MEDICALRECORD | → `[SSN]` etc. |
| ORGANIZATION | → `[ORGANIZATION]` |
| USERNAME | → `[USERNAME]` |